**Project 2**

**Recommender System Using TFIDF**

In [ ]:
#desired libraries
#this project is about getting the top 5 similar movies closed to this movie based on distance
#we can measure distance in two ways first cosine similarity which is recommended , second is euclidean distance
#so we apply nlp to strings related to each movie so we can numerize words and get the clothest based on score
#we will need at most two columns to have enough information
#converting these columns to BOW method and here we will use TFIDF , we can use count vectorizer
#second we apply similarity score using cosine similarity or euclidean disatnce
#this project is clustering not classification so there is no output and here we donot need any classical machine learning algorithm
#basic numerical computing libraries
import numpy as np
import pandas as pd
#import tfidf model
from sklearn.feature_extraction.text import TfidfVectorizer
#import libararies of similarities
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

In [ ]:
#import data
#better download it as csv from the lazy programmer website and import it locally
raw_data = pd.read_csv('tmdb_5000_movies.csv')
df = raw_data.copy()
df.sample(5)

**cleansing**

In [ ]:
#investing
df.shape
#here we have 20 columns with 20 features and all we need is 2 columns to cluster based on title column

In [ ]:
#check numerical features
df.describe().T

In [ ]:
#check categorical features
df.describe(include="O").T

In [ ]:
df.dtypes

In [ ]:
#get general overview
df.info()

In [ ]:
#check duplicates
df.duplicated().sum()

In [ ]:
#here i care about three columns ---> (title , genre , keywords)
df_f = df[['title','genres','keywords']]
df_f.head()

In [ ]:
#check missing value
#there is no missing value
df_f.isna().sum()

In [ ]:
#check columns
#rename columns
df_f.columns=df_f.columns.str.lower().str.strip().str.replace(" ","_")
df_f.columns

In [ ]:
#check object features
#no object features exist
for i in  df_f.select_dtypes('object'):
    print(f'for {i} : \n',df_f[i].value_counts())
    print('-'*60)

In [ ]:
df_f.info()

In [ ]:
#here we donot have any stop words to remove and no need to stemming or lemmatization

**Text Preprocessing**

In [ ]:
#we notice that the inputs are in json so i need to extarct the whole text in these two columns and make it in one single column called string which i will build my clustering on
import json
df_f['genres']

In [ ]:
df_f['keywords']

In [ ]:
x=df_f.iloc[0]
x

In [ ]:
j=json.loads(x['genres'])
j

In [ ]:
stri=[]
for json in j :
   stri.append(json['name'])
' '.join(stri)

In [ ]:
import json
#function to extact information and returns it in one single column
#we pass row each time in this function 
def create_string(row):
    #here we pass the final data frame and apply some preprocessing on it 
    #we do two phases one for genres and the other for keywords
    #we must load json
    #then gather all string in single variable
    
    #phase 1 'genres'
    j_1=json.loads(row['genres'])
    stri=[]
    for j in j_1 :
        stri.append(j['name'])
    genres=' '.join(stri)
    
    
    #phase 2 'keywords'
    j_2=json.loads(row['keywords'])
    stri=[]
    for j in j_2 :
        stri.append(j['name'])
    keywords=' '.join(stri)
    
    #return genres and keywords text together into single string
    return genres+" "+keywords

#add new column called string and apply this function to the whole data frame with axis = 1 beacuse we work on more than one column which means that it is data set so we have to sepicify wether it is column or row
df_f['string'] = df_f.apply(create_string,axis=1)    

In [ ]:
#general overview
df_f

**Bag of Words**

**TFIDF**

In [ ]:
#create tfidf instance
tfidf = TfidfVectorizer()
#apply the model on string column to get similarity later based on this x when passing query
x = tfidf.fit_transform(df_f['string'])

In [ ]:
x

**Testing on query**

In [ ]:
#let's say we have Scream 3 movie and i want to get similar movies based on string column so i have to search in x database
#first get the index of movie from df_f
#apply filter
movie = df_f[df_f['title'] == 'Scream 3']
movie

In [ ]:
#second get the index with 0 to get the number directly without needing to make new data frame mapping each movie to index number
idx = movie.index[0]
idx

In [ ]:
#third pass this index to x to access it 
#make sure that index is the number of row in x which is related to the spicific movie in our sparse matrix
query = x[idx]
query

In [ ]:
#fourth convert it to vector so we can compare it with other vectors to get similarity
arr = query.toarray()
arr

In [ ]:
#fifth compute similarity between this vector and others so we can get similar movies 
#we will use cosine similarity
scores = cosine_similarity(arr ,x)
scores

In [ ]:
#0 means dissimilar and 1 means similar
#sixth we flatten our scores so we can have it as pd instead if 1 x N matrix
scores = scores.flatten()
scores

**EDA**

In [ ]:
#we choose plot library because i will visualize scores only and they are pd of numbers not related to any data frame
import matplotlib.pyplot as plt
plt.plot(scores)

In [ ]:
#seventh sort the scores from dissimilar to similar
scores.argsort()

In [ ]:
#display the scores from smallest to largest so we apply .argsort() function to make sure about replacement
plt.plot(scores[scores.argsort()])

In [ ]:
#so we want the versa order and versa disaplay from largest to smallest
#so we say - scores
(-scores).argsort()

In [ ]:
#display the scores from largest to smallest so we apply .argsort() function to make sure about replacement
plt.plot(scores[(-scores).argsort()])

In [ ]:
#eighth get recommended movies
#first get index 
#we start from 1 as we donot want the zero index which is related to 1 and it is the movie it self
#top 5
#i canot apply .head() because it will be the movie itself
idx_list=(-scores).argsort()[1:6]
idx_list

In [ ]:
#second apss it to data fram title column
#then convert it to list so we can store the name of the movies in list
df_f['title'].iloc[idx_list].to_list()


**Validation**

In [ ]:
title = input('Enter Movie exist in our data base : ')
def get_recommend (title):
    try :
        #first get the index of movie from df_f
        #apply filter
        #to capatalize every word in string in case the user forget
        title = ' '.join(word.capitalize() for word in title.split())
        movie = df_f[df_f['title'] == title]
        #second get the index with 0 to get the number directly without needing to make new data frame mapping each movie to index number
        idx = movie.index[0]
        #third pass this index to x to access it 
        #make sure that index is the number of row in x which is related to the spicific movie in our sparse matrix
        query = x[idx]
        #fourth convert it to vector so we can compare it with other vectors to get similarity
        arr = query.toarray()
        #fifth compute similarity between this vector and others so we can get similar movies 
        #we will use cosine similarity
        scores = cosine_similarity(arr ,x)  
        #0 means dissimilar and 1 means similar
        #sixth we flatten our scores so we can have it as pd instead if 1 x N matrix
        scores = scores.flatten()
        #seventh get recommended movies
        #first get index 
        #we start from 1 as we donot want the zero index which is related to 1 and it is the movie it self
        #top 5
        #i canot apply .head() because it will be the movie itself
        idx_list=(-scores).argsort()[1:6]
        #second apss it to data fram title column
        #then convert it to list so we can store the name of the movies in list
        return df_f['title'].iloc[idx_list].to_list()
    except :
        return 'Sorry! movie not found in data base'
print(f'Recommended movies for {title} : ',get_recommend(title))